In [1]:
import optuna
import numpy as np
import joblib

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score


In [2]:
RESULTS_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\notebooks\results"

X_train = np.load(f"{RESULTS_DIR}/X_train_cnnvit.npy")
y_train = np.load(f"{RESULTS_DIR}/y_train_cnnvit.npy")

X_val = np.load(f"{RESULTS_DIR}/X_val_cnnvit.npy")
y_val = np.load(f"{RESULTS_DIR}/y_val_cnnvit.npy")

X_test = np.load(f"{RESULTS_DIR}/X_test_cnnvit.npy")
y_test = np.load(f"{RESULTS_DIR}/y_test_cnnvit.npy")

print("Data loaded ")


Data loaded 


In [3]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 700),
        "max_depth": trial.suggest_int("max_depth", 4, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 6),
        "gamma": trial.suggest_float("gamma", 0.0, 0.3),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 0.3),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 2.0),
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1
    }

    model = XGBClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds = model.predict(X_val)
    return accuracy_score(y_val, preds)


In [4]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best Validation Accuracy:", study.best_value)
print("Best Params:", study.best_params)


[I 2026-02-13 19:10:07,732] A new study created in memory with name: no-name-9d76d7ed-b04b-44ad-a64f-37c03602a296
[I 2026-02-13 19:12:08,727] Trial 0 finished with value: 0.9322766570605188 and parameters: {'n_estimators': 637, 'max_depth': 9, 'learning_rate': 0.02161501593555079, 'subsample': 0.6159484213028942, 'colsample_bytree': 0.6649414326070925, 'min_child_weight': 4, 'gamma': 0.2645986881127846, 'reg_alpha': 0.22128257469612211, 'reg_lambda': 1.257480222823216}. Best is trial 0 with value: 0.9322766570605188.
[I 2026-02-13 19:12:49,792] Trial 1 finished with value: 0.9438040345821326 and parameters: {'n_estimators': 367, 'max_depth': 5, 'learning_rate': 0.07729884038192995, 'subsample': 0.7404830211638564, 'colsample_bytree': 0.8887272338619774, 'min_child_weight': 6, 'gamma': 0.024977127222031003, 'reg_alpha': 0.2162615280594087, 'reg_lambda': 0.5373288594242732}. Best is trial 1 with value: 0.9438040345821326.
[I 2026-02-13 19:14:32,204] Trial 2 finished with value: 0.9236311

Best Validation Accuracy: 0.9438040345821326
Best Params: {'n_estimators': 367, 'max_depth': 5, 'learning_rate': 0.07729884038192995, 'subsample': 0.7404830211638564, 'colsample_bytree': 0.8887272338619774, 'min_child_weight': 6, 'gamma': 0.024977127222031003, 'reg_alpha': 0.2162615280594087, 'reg_lambda': 0.5373288594242732}


In [5]:
best_params = study.best_params

best_model = XGBClassifier(
    **best_params,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)

print("Optimized model trained ")


Optimized model trained 


In [7]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

TPR = tp / (tp + fn)   # Sensitivity / Recall
FPR = fp / (fp + tn)

print("===== CNN + ViT + XGB (After Optuna) =====")
print("TPR (Sensitivity):", TPR)
print("FPR:", FPR)


===== CNN + ViT + XGB (After Optuna) =====
TPR (Sensitivity): 0.9234449760765551
FPR: 0.06044678055190539


In [6]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("\n===== CNN + ViT + XGB (After Optuna) =====")
print("Test Accuracy:", acc)
print("Test AUROC:", auc)



===== CNN + ViT + XGB (After Optuna) =====
Test Accuracy: 0.9322766570605188
Test AUROC: 0.9830764942460081
